# VLM-DENTAL — Stage 1: Supervised Fine-Tuning (SFT)

This notebook trains **Qwen/Qwen3.5-9B** using explicit **BF16 LoRA or NF4 QLoRA** profiles ($r=32, \alpha=64$) on verified expert dental clinical traces across 3 staged curriculum milestones.

### Core Architectural & Clinical Invariants:
- **3-Stage Curriculum with Negative Controls Calibration**:
  - **Stage 1a (`dentex_alone`)**: 678 DENTEX disease traces + 27 healthy DENTEX negative controls $\rightarrow$ `qwen3_5_9b_sft_{track}_dentex`.
  - **Stage 1b (`dentex_tufts_overlap`)**: DENTEX + Tufts overlapping disease (caries, periapical) + Healthy controls (DENTEX + Tufts) $\rightarrow$ `qwen3_5_9b_sft_{track}_dentex_tufts_overlap`.
  - **Stage 1c (`multicohort_all`)**: DENTEX (4 findings) + Tufts Full (all 4 findings on their own) + Full healthy controls $\rightarrow$ `qwen3_5_9b_sft_{track}_multicohort_all`.
- **Multimodal Vision Projector LoRA**: Adapts `merger.mlp.0` and `merger.mlp.2` alongside LLM attention/MLP projections, specializing radiographic feature projection without catastrophic forgetting of base ViT representations.
- **Canonical Vision Views**: the model is shown fixed-size, aspect-preserving views of the radiograph and tool outputs while every tool still runs on the native image. On TPU, static vision-slot padding gives XLA a single graph; on GPU samples are padded dynamically. Details: `docs/CANONICAL_VISION_AND_TPU_PIPELINE.md`.
- **Hardware**: single-process SPMD on the 8 TPU cores, or one CUDA GPU. Precision is exactly what `PRECISION` says (`bf16` = LoRA, `qlora` = LoRA on 4-bit weights); nothing is switched automatically.
- **Strict Track Segregation**:
  - **Track A (`with_tools`)**: Multi-turn agent trained on real workstation tool-use traces (8 tools).
  - **Track B (`no_tools`)**: Single-turn direct radiologist trained on tool-free Chain-of-Thought (CoT) traces.
- **Cosine Warmup & Gradient Clipping**: 5% linear warmup, cosine decay to $1\times 10^{-6}$, and gradient norm clipping at $1.0$.
- **Validation Checkpointing**: 5% held-out validation split with `best_adapter` retention.
- **Hugging Face Hub Checkpoint Sync**: Checkpoints (~760 MB LoRA + optimizer) are pushed every 25 steps to survive Kaggle 9-hour session limits and enable seamless multi-account resume.

### Consolidated hardware repair
Select `HARDWARE_PROFILE` in configuration: `quadro_rtx8000`, `dual_t4`, `rtx4090_qlora`, `workstation48_lora`, or `tpu`. The GPU profiles start with a two-window debug run; set `DEBUG_STEPS=0` for normal training after verification. Larger chunk sizes are throughput candidates, not verified VRAM guarantees. See `docs/SFT_GPU_RUNBOOK.md` and `docs/SFT_CHUNKED_LOSS.md`.

The bootstrap installs shared dependencies first. On Kaggle dual T4 only, a later hardware-aware cell loads the prebuilt causal-conv1d wheel from `rezanadimi/causal-conv1d-t4-wheelhouse`; other CUDA profiles compile for their active GPU architecture, and TPU skips the CUDA extension. Existing image/model datasets under `rezanadimikj` are unchanged.

### Checkpoint upload repair (0043)

The settings cell separates local `MODEL_ID` from Hub `HF_BASE_MODEL_ID`, exposes an optional `HF_RUN_TAG`, and backs up every five optimizer steps plus epoch boundaries. TPU loss/probe settings are passed to training and AOT separately from the GPU chunk flag.

**Before Commit & Run:** apply patch 0043 to the repository this notebook clones and push that source yourself, or include a patch-application cell after cloning. Importing this notebook does not update the cloned Python scripts. See `docs/HUB_CHECKPOINTS.md` for recovery uploads and patch choices.


## 1. Platform Detection & Shallow Repository Clone

Detects runtime environment (Kaggle vs Colab vs Local) and performs a shallow clone (`--depth 1`) to eliminate history download overhead.

In [8]:
import os
import sys
from pathlib import Path

# 1. Detect platform environment
IS_KAGGLE = os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
PLATFORM_NAME = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "Local PC / Server")
print(f"[PLATFORM] Detected Runtime Environment: {PLATFORM_NAME}")

# 2. Find the repo root (the folder named VLM-DENTAL) before ever cloning
REPO_NAME = "VLM-DENTAL"
REPO_URL = "https://github.com/rezaxr14/VLM-DENTAL.git"

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if p.name == REPO_NAME), None)  # cwd or any parent (notebooks/)

if repo_root is None and (cwd / REPO_NAME).is_dir():
    repo_root = cwd / REPO_NAME                                                   # repo is a subfolder of cwd

if repo_root is None:
    print(f"[CLONE] No local repo found. Shallow cloning {REPO_URL}...")
    !git clone --depth 1 {REPO_URL}
    repo_root = cwd / REPO_NAME
    if not repo_root.is_dir():
        raise RuntimeError("git clone failed, check network / URL / permissions")

os.chdir(repo_root)
print(f"[WORKSPACE] Using repository root: {Path.cwd()}")

[PLATFORM] Detected Runtime Environment: Local PC / Server
[WORKSPACE] Using repository root: /home/nadimi/VLM-DENTAL


## 2. Shared Dependency Installation & Environment Setup

Installs shared packages without replacing Kaggle's PyTorch/CUDA runtime. After hardware-profile selection, dual T4 loads the prebuilt causal-conv1d wheelhouse dataset, other CUDA GPUs compile their own extension, and TPU skips the CUDA extension.


In [9]:
# # Install shared VLM-DENTAL dependencies.
# # Hardware-specific CUDA dependencies are installed after HARDWARE_PROFILE is selected.
# # Do not upgrade or replace Kaggle's preinstalled PyTorch/CUDA runtime.
# !pip uninstall -y torchao
# !pip install -q -U "transformers>=5.0.0" accelerate peft trl datasets huggingface_hub ultralytics python-dotenv qwen-vl-utils tabulate tpu-info
# !pip install -q --no-deps -e .


## 3. Hardware & Cloud TPU v5e-8 Topology Detection

Initializes PyTorch/XLA on TPU v5e-8 (reporting 8-way distributed TPU device count and chip topology) or reports CUDA GPUs.

In [10]:
import os
import torch

IS_TPU = False
DEVICE_STR = "cpu"

# Detect TPU availability without initializing the PJRT hardware client in the
# notebook kernel. Calling xm.xla_device() in the notebook kernel locks /dev/vfio/*
# hardware devices, which prevents child training scripts (!python scripts/train_sft.py)
# from spawning multi-core workers with 'open(/dev/vfio/1): Device or resource busy'.
if os.path.exists("/dev/vfio") or os.environ.get("PJRT_DEVICE") == "TPU" or "COLAB_TPU_ADDR" in os.environ:
    try:
        import torch_xla
        IS_TPU = True
        DEVICE_STR = "Cloud TPU (8 cores / chips via PyTorch/XLA)"
        print(f"[HARDWARE] SUCCESS: Detected {DEVICE_STR}")
        print(f"[HARDWARE] 8-Way PyTorch/XLA GSPMD (Single-Process Multi-Device) ready across 128 GB total HBM (~2.3 GB / core).")
        
        # TPU attention fusions are managed natively via PyTorch SDPA (--attn-implementation sdpa)
        print('[HARDWARE] Attention optimization configured via native PyTorch SDPA on TPU.')
    except ImportError:
        pass

if not IS_TPU:
    if torch.cuda.is_available():
        gpu_count = torch.cuda.device_count()
        gpu_name = torch.cuda.get_device_name(0)
        DEVICE_STR = f"CUDA ({gpu_count}x {gpu_name})"
        print(f"[HARDWARE] SUCCESS: Detected {DEVICE_STR}")
    else:
        print("[HARDWARE] Running on CPU / Standard environment")

print(f"[FRAMEWORK] PyTorch: {torch.__version__}")

[HARDWARE] SUCCESS: Detected CUDA (1x Quadro RTX 8000)
[FRAMEWORK] PyTorch: 2.13.0+cu126


In [11]:
# %%writefile /kaggle/working/VLM-DENTAL/.env

## 4. Secrets Diagnostics, Hub Auth & Verified Traces Sync

Audits environment variables from `.env`, Kaggle Secrets Vault, and Colab Userdata, authenticates with Hugging Face Hub, and synchronizes verified clinical trace splits.

In [12]:
from dotenv import load_dotenv
from huggingface_hub import login, snapshot_download
from tabulate import tabulate

# 1. Load local .env if present
load_dotenv()

# Helper for Kaggle Secrets / Colab Userdata ingestion
def get_secret_multisource(key: str, default: str | None = None) -> tuple[str | None, str]:
    val = os.environ.get(key)
    if val:
        return val, ".env / OS Env"

    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            sec = UserSecretsClient().get_secret(key)
            if sec:
                os.environ[key] = sec
                return sec, "Kaggle Secrets"
        except Exception:
            pass

    if IS_COLAB:
        try:
            from google.colab import userdata
            sec = userdata.get(key)
            if sec:
                os.environ[key] = sec
                return sec, "Colab Userdata"
        except Exception:
            pass

    if default is not None:
        os.environ[key] = default
        return default, "Default Fallback"

    return None, "Not Set"

def mask_secret(val: str | None, is_secret: bool = True) -> str:
    if not val:
        return "---"
    if not is_secret:
        return val
    if len(val) <= 8:
        return "********"
    return f"{val[:4]}...{val[-4:]}"

# Audit list of project environment variables & tokens
TOKEN_SPECS = [
    ("HF_TOKEN", True, None),
    ("HF_ARTIFACT_REPO", False, "Reza-Nadimi/vlm-dental-models"),
    ("HF_TRACES_REPO", False, "Reza-Nadimi/vlm-dental-traces"),
    ("DENTEX_IMAGES_REPO", False, "Reza-Nadimi/dentex-train-images"),
    ("TUFTS_IMAGES_REPO", False, "Reza-Nadimi/tufts-train-images"),
    ("OPENAI_API_KEY", True, None),
    ("ANTHROPIC_API_KEY", True, None),
    ("GEMINI_API_KEY", True, None),
    ("GROQ_API_KEY", True, None),
    ("NVIDIA_API_KEY", True, None),
    ("WANDB_API_KEY", True, None),
    ("DENTAL_AGENT_DATA_DIR", False, "data"),
]

status_table = []
for key, is_sec, def_val in TOKEN_SPECS:
    val, source = get_secret_multisource(key, def_val)
    status_str = "[SET]" if val else "[NOT SET]"
    status_table.append([key, status_str, source, mask_secret(val, is_sec)])

print("=" * 80)
print("VLM-DENTAL: ENVIRONMENT & AUTHENTICATION DIAGNOSTICS")
print("=" * 80)
print(tabulate(status_table, headers=["Environment Variable", "Status", "Detection Source", "Configured / Masked Value"], tablefmt="fancy_grid"))

# 2. Authenticate Hugging Face Hub
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    try:
        login(token=hf_token, add_to_git_credential=True)
        print("\n[AUTH] Successfully authenticated with Hugging Face Hub.")
    except Exception as e:
        print(f"\n[AUTH WARNING] Hugging Face login failed: {e}")
else:
    print("\n[AUTH WARNING] No HF_TOKEN detected. Checkpoint upload will be disabled unless logged in interactively:")
    login()

# 3. Synchronize verified clinical trace splits from Hugging Face Hub
print("\n[SYNC] Synchronizing verified clinical trace splits from Hugging Face...")
!python scripts/sync_traces_hf.py --download

# Verify that token length manifest was downloaded alongside traces
manifest_p = Path("data/traces/trace_token_lengths.json")
if manifest_p.exists():
    print(f"[SYNC] trace_token_lengths.json verified ({manifest_p.stat().st_size / 1024:.1f} KB).")
else:
    print("[SYNC NOTICE] trace_token_lengths.json not yet found on Hub. Traces will load without manifest masking until uploaded.")

# =========================================================================
# 4. PANORAMIC RADIOGRAPHIC IMAGES INGESTION (DUAL-PATH ARCHITECTURE)
# =========================================================================
# Path A (Attached Kaggle Dataset): Auto-detects /kaggle/input/dentex-panoramic
#        and /kaggle/input/tufts-panoramic if attached to this session.
#        (0 download time, 0 MB disk space used from your 40 GB workspace quota).
# Path B (Traditional Hugging Face Hub): Downloads directly from Hugging Face Hub
#        ('Reza-Nadimi/dentex-train-images' and 'Reza-Nadimi/tufts-train-images').
#
# TOGGLE: Set FORCE_HF_DATASET_DOWNLOAD = True to force remote Hugging Face
# download even when Kaggle inputs are attached.
FORCE_HF_DATASET_DOWNLOAD = False

# --- A. DENTEX Ingestion ---
dentex_local = Path("data/dentex")
dentex_local.mkdir(parents=True, exist_ok=True)
kaggle_dentex_candidates = [
    Path("/kaggle/input/datasets/rezanadimikj/dentex-panoramic"),
    Path("/kaggle/input/dentex-panoramic"),
    Path("/kaggle/input/dentex"),
    Path("/home/asr/Reza_Nadimi/VLM-DENTAL/data/dentex"),
    Path("/home/nadimi/VLM-DENTAL/data/dentex"),
]
dentex_mounted = False

if not FORCE_HF_DATASET_DOWNLOAD:
    for cand in kaggle_dentex_candidates:
        if cand.exists():
            target_img_dir = cand / "images" if (cand / "images").exists() else cand
            link_target = dentex_local / "images"
            if not link_target.exists():
                try:
                    link_target.symlink_to(target_img_dir)
                except Exception:
                    pass
            print(f"[DATASET] Attached Kaggle DENTEX detected at {cand} (Skipping remote download).")
            dentex_mounted = True
            break

if not dentex_mounted:
    dentex_repo = os.environ.get("DENTEX_IMAGES_REPO", "Reza-Nadimi/dentex-train-images")
    has_dentex = (dentex_local / "images").exists() or (dentex_local / "train_images").exists() or Path("data/images").exists()
    if dentex_repo and not has_dentex:
        print(f"\n[SYNC] Downloading DENTEX panoramic images from Hugging Face ({dentex_repo})...")
        try:
            snapshot_download(
                repo_id=dentex_repo,
                repo_type="dataset",
                local_dir=str(dentex_local),
                token=hf_token,
            )
            print("[SYNC] DENTEX images ready.")
        except Exception as e:
            print(f"[SYNC WARNING] DENTEX images download failed: {e}")
    else:
        print("\n[SYNC] DENTEX panoramic images verified on local disk.")

# --- B. Tufts Ingestion ---
tufts_local = Path("data/Tufts")
tufts_local.mkdir(parents=True, exist_ok=True)
kaggle_tufts_candidates = [
    Path("/kaggle/input/datasets/rezanadimikj/tufts-panoramic"),
    Path("/kaggle/input/tufts-panoramic"),
    Path("/kaggle/input/tufts"),
    Path("/home/asr/Reza_Nadimi/VLM-DENTAL/data/Tufts"),
    Path("/home/nadimi/VLM-DENTAL/data/Tufts"),
]
tufts_mounted = False

if not FORCE_HF_DATASET_DOWNLOAD:
    for cand in kaggle_tufts_candidates:
        if cand.exists():
            target_img_dir = cand / "Radiographs" if (cand / "Radiographs").exists() else cand
            link_target = tufts_local / "Radiographs"
            if not link_target.exists():
                try:
                    link_target.symlink_to(target_img_dir)
                except Exception:
                    pass
            print(f"[DATASET] Attached Kaggle Tufts detected at {cand} (Skipping remote download).")
            tufts_mounted = True
            break

if not tufts_mounted:
    tufts_repo = os.environ.get("TUFTS_IMAGES_REPO", "Reza-Nadimi/tufts-train-images")
    has_tufts = (tufts_local / "Radiographs").exists() or (tufts_local / "images").exists() or Path("data/Tufts/Radiographs").exists()
    if tufts_repo and not has_tufts:
        print(f"\n[SYNC] Downloading Tufts panoramic images from Hugging Face ({tufts_repo})...")
        try:
            snapshot_download(
                repo_id=tufts_repo,
                repo_type="dataset",
                local_dir=str(tufts_local),
                token=hf_token,
            )
            print("[SYNC] Tufts images ready.")
        except Exception as e:
            print(f"[SYNC WARNING] Tufts images download failed: {e}")
    else:
        print("[SYNC] Tufts panoramic images verified on local disk.")


VLM-DENTAL: ENVIRONMENT & AUTHENTICATION DIAGNOSTICS
╒════════════════════════╤═══════════╤════════════════════╤═════════════════════════════════╕
│ Environment Variable   │ Status    │ Detection Source   │ Configured / Masked Value       │
╞════════════════════════╪═══════════╪════════════════════╪═════════════════════════════════╡
│ HF_TOKEN               │ [SET]     │ .env / OS Env      │ hf_I...JcGM                     │
├────────────────────────┼───────────┼────────────────────┼─────────────────────────────────┤
│ HF_ARTIFACT_REPO       │ [SET]     │ .env / OS Env      │ Reza-Nadimi/vlm-dental-models   │
├────────────────────────┼───────────┼────────────────────┼─────────────────────────────────┤
│ HF_TRACES_REPO         │ [SET]     │ .env / OS Env      │ Reza-Nadimi/vlm-dental-traces   │
├────────────────────────┼───────────┼────────────────────┼─────────────────────────────────┤
│ DENTEX_IMAGES_REPO     │ [SET]     │ .env / OS Env      │ Reza-Nadimi/dentex-train-images │
├──────

Token has not been saved to git credential helper.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.

[AUTH] Successfully authenticated with Hugging Face Hub.

[SYNC] Synchronizing verified clinical trace splits from Hugging Face...

HUGGING FACE TRACE DOWNLOAD
Source Repo: Reza-Nadimi/vlm-dental-traces (dataset)
Target Dir : data/traces
[EXISTS] train_cot_traces.jsonl (25.72 MB) already present. Skipping.
[EXISTS] train_cot_traces_no_tools.jsonl (8.83 MB) already present. Skipping.
[EXISTS] train_cot_traces_dentex.jsonl (20.54 MB) already present. Skipping.
[EXISTS] train_cot_traces_dentex_no_tools.jsonl (7.19 MB) already present. Skipping.
[EXISTS] train_cot_traces_tufts.jsonl (5.17 MB) already p

## 5. Interactive SFT Configuration & Curriculum Stage Selection



In [13]:
from tabulate import tabulate

# =========================================================================
# STAGE 1 SFT EXECUTION PARAMETERS
# =========================================================================
# Curriculum Stage:
#   - 'dentex_alone'          : Stage 1a: DENTEX Alone + DENTEX Healthy Controls
#   - 'dentex_tufts_overlap'  : Stage 1b: DENTEX + Tufts Overlap + Negative Controls
#   - 'multicohort_all'       : Stage 1c: Full Multi-Cohort: DENTEX + Tufts All 4 Findings + Full Negative Controls
# =========================================================================
# BASE MODEL: ONE local folder, found by dental_agent.utils.inputs.resolve_qwen_model() (nothing is copied or moved)
# =========================================================================
# First valid candidate wins:
#   1. env QWEN35_9B_LOCAL_DIR (set but unusable = an error)
#   2. attached Kaggle model / dataset, read-only and in place (Colab: <persist dir>/models/Qwen3.5-9B on Drive)
#   3. <repo>/data/models/Qwen3.5-9B   (canonical folder, found from the repo, not from the working directory)
#   4. an existing Hugging Face cache snapshot of Qwen/Qwen3.5-9B
#   5. none found: download into 3 (resumes a partial copy; offline = an error that says where to put the model)
# A folder counts only with config.json, tokenizer + processor files and every weight shard present and non-empty.
# The Hub id is never handed to from_pretrained. TOGGLE: FORCE_HF_MODEL_DOWNLOAD = True skips 1-4 and (re)downloads into 3.
FORCE_HF_MODEL_DOWNLOAD = False
from dental_agent.utils.inputs import LOCAL_BASE_MODEL_ENV, resolve_qwen_model
MODEL_ID, MODEL_SOURCE = resolve_qwen_model(force_download=FORCE_HF_MODEL_DOWNLOAD)   # prints: Qwen3.5-9B source: <source> -> <path>
os.environ[LOCAL_BASE_MODEL_ENV] = MODEL_ID   # later cells and subprocesses (evaluation, zero-shot cache lookup) follow this same folder

STAGE = "multicohort_all"

# Training Track:
#   - 'with_tools' : Multi-turn diagnostic agent with 8 workstation tools
#   - 'no_tools'   : Direct radiologist (tool-free Chain-of-Thought)
TRACK = "with_tools"

# Vision LoRA Mode: 'projector' (the merger Linear layers, merger.linear_fc1 / merger.linear_fc2) or 'none'
LORA_TARGET_VISION = "projector"   # --lora-target-vision: LoRA on the vision projector (the merger Linear layers)
MASK_MALFORMED_TURNS = True        # --mask-malformed-turns: rejected assistant turns stay as context, not as targets
# Dev split: traces of these DENTEX train images are held out of SFT (and GRPO) for the K sweep and ablations.
# Create it once:  !python scripts/make_dev_split.py --data-dir data --n 70 --seed 42   (commit data/splits/dentex_dev.json)
DEV_SPLIT = "data/splits/dentex_dev.json"   # --dev-split
LORA_LINEAR_ATTN = True            # --lora-linear-attn: LoRA on the Gated-DeltaNet layers (in_proj_qkv, in_proj_z, out_proj)

# Explicit profile selection; no GPU model name guesses or silent precision changes.
# GPU profiles preserve TARGET_SEQ_LEN=16384 and the established image geometry; the tpu profile uses 32768 (see TARGET_SEQ_LEN below).
# dual_t4: 128-token chunks are observed working; 256 below is the requested larger candidate, test first.
HARDWARE_PROFILE = "quadro_rtx8000"  # quadro_rtx8000 | dual_t4 | rtx4090_qlora | workstation48_lora | tpu
_PROFILES = {
    "dual_t4": dict(hub_tag="dual_t4_fp16", resume=True, precision="qlora", compute="fp16", chunk=256, attention="sdpa",
                    repeat_kv="on", loader="staged", load_threads=1, efficient_sdpa=True),
    "rtx4090_qlora": dict(hub_tag="rtx4090_bf16", resume=True, precision="qlora", compute="bf16", chunk=1024, attention="auto",
                         repeat_kv="auto", loader="standard", load_threads=0, efficient_sdpa=True),
    "workstation48_lora": dict(hub_tag="workstation48_lora_bf16", resume=True, precision="bf16", compute="auto", chunk=2048, attention="auto",
                              repeat_kv="auto", loader="standard", load_threads=0, efficient_sdpa=True),
    "quadro_rtx8000": dict(hub_tag="quadro_rtx8000_fp16", resume=True, precision="fp16", compute="auto", chunk=2048, attention="sdpa",
                            repeat_kv="auto", loader="standard", load_threads=0, efficient_sdpa=True),
    "tpu": dict(hub_tag="tpu_bf16", resume=True, precision="bf16", compute="auto", chunk=0, attention="sdpa",
                repeat_kv="off", loader="standard", load_threads=0, efficient_sdpa=False),
}
_profile = _PROFILES[HARDWARE_PROFILE]
if (HARDWARE_PROFILE == "tpu") != IS_TPU:
    raise RuntimeError("Select a profile that matches this runtime; precision will not switch automatically")
PRECISION = _profile["precision"]
QLORA_COMPUTE_DTYPE = _profile["compute"]  # only used with PRECISION='qlora'
LOSS_CHUNK_TOKENS = _profile["chunk"]
ATTN_IMPLEMENTATION = _profile["attention"]
SDPA_REPEAT_KV = _profile["repeat_kv"]
LOADER_MODE = _profile["loader"]
LOAD_CPU_THREADS = _profile["load_threads"]
REQUIRE_EFFICIENT_SDPA = _profile["efficient_sdpa"]
LOAD_TRACEBACK_SECONDS = 0  # set 60 only when investigating a loading stall
DEBUG_STEPS = 0  # 0 = normal training; GPU debug uses accumulation 1
RUN_ATTENTION_PROBE = HARDWARE_PROFILE != "tpu"
# Separate static TPU loss; leave the GPU chunk/profile values above unchanged.
TPU_LOSS_CHUNK_TOKENS = 256 if IS_TPU else 0
RUN_TPU_LOSS_PROBE = IS_TPU and TPU_LOSS_CHUNK_TOKENS > 0
TPU_LOSS_PROBE_PASSED = None

# Hint only -- the run continues with PRECISION exactly as set above.
if not IS_TPU and torch.cuda.is_available() and PRECISION != "qlora":
    _vram_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
    if _vram_gib < 40:
        print(f"[HINT] {torch.cuda.get_device_name(0)} has {_vram_gib:.0f} GiB; a 9B model in {PRECISION} LoRA usually "
              "does not fit. Consider PRECISION = 'qlora'. Continuing with PRECISION = '" + PRECISION + "'.")

# Hardware & Multi-Core Distribution (Cloud TPU v5e-8)
NUM_CORES = 8 if IS_TPU else 1
USE_FSDP = True if IS_TPU else False

# Hyperparameters
DATA_DIR = "data"
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 1 if (IS_TPU and NUM_CORES > 1) else 8
EPOCHS = 3
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# Evaluation & Checkpointing Strategies
# Strategy options:
#   - 'epoch' : Run evaluation/checkpoint save at end of each completed epoch (Recommended & Default)
#   - 'steps' : Run evaluation/checkpoint save at step intervals (--eval-every-steps / --push-every-steps)
#   - 'both'  : Run at both step intervals and at epoch boundaries
EVAL_STRATEGY = "epoch"
EVAL_EVERY_STEPS = 0  # 0 = epoch-boundary only; > 0 = intermediate step evaluations

SAVE_STRATEGY = "both"
PUSH_EVERY_STEPS = 20  # Back up every 5 optimizer steps as well as each epoch; includes resume state.

# Checkpoint Sync & Kaggle Continuity
HF_REPO = os.environ.get("HF_ARTIFACT_REPO", "Reza-Nadimi/vlm-dental-models")
HF_BASE_MODEL_ID = "Qwen/Qwen3.5-9B"  # Metadata only: MODEL_ID still loads the attached/local folder.
HF_RUN_TAG = _profile["hub_tag"]  # Upload and resume always use the same profile namespace.
HF_UPLOAD_CHECK = True  # Tiny authenticated write before loading 9B weights (not during read-only preflight).
RESUME = _profile["resume"]  # Resume only this profile, track, stage and precision; absent checkpoint starts fresh.
# Run the doctor first (the launch cell re-runs this exact command with --preflight; read-only, loads no model): every input is
# found (local, Kaggle input, Hugging Face Hub) and reported PASS / WARN / FAIL. A FAIL stops the cell before the model loads.
PREFLIGHT_FIRST = True

# Hardware Memory Wall Bypasses
XLA_PALLAS = True   # Enables TPU attention compilation and kernel optimizations (SDPA lowering)
XLA_SPMD = True     # Single-process PyTorch/XLA GSPMD (True: recommended, saves host RAM) vs legacy xmp.spawn (False)

# XLA Persistent Compilation Cache (Auto-detected if attached as Kaggle input or in data/xla_cache)
def resolve_xla_cache_dir_notebook() -> str | None:
    def is_valid_cache(d: Path) -> bool:
        if not d.is_dir():
            return False
        # Valid cache must contain compilation artifacts other than just dataset-metadata.json
        files = [f for f in d.iterdir() if f.name != "dataset-metadata.json"]
        return len(files) > 0

    candidates = [
        Path("/kaggle/input/datasets/rezanadimikj/vlm-dental-xla-cache"),
        Path("/kaggle/input/datasets/rezanadimikj/vlm-dental-xla-cache/xla_cache"),
        Path("/kaggle/input/vlm-dental-xla-cache"),
        Path("/kaggle/input/vlm-dental-xla-cache/xla_cache"),
        Path("/kaggle/input/xla-cache-sft"),
        Path("/kaggle/working/xla_cache"),
        Path("data/xla_cache"),
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for item in sorted(kaggle_input.glob("**/datasets/*/*xla*cache*")):
            if item.is_dir():
                candidates.insert(0, item)
        for item in sorted(kaggle_input.glob("**/*xla*cache*")):
            if item.is_dir():
                candidates.insert(0, item)
        for item in sorted(kaggle_input.glob("*xla*cache*")):
            if item.is_dir():
                candidates.insert(0, item)

    for cand in candidates:
        target = cand / "xla_cache" if (cand / "xla_cache").is_dir() else cand
        if is_valid_cache(target):
            artifact_count = len(list(target.iterdir()))
            print(f"[XLA CACHE] Detected persistent compilation cache ({artifact_count} items): {target}")
            return str(target)

    print("[XLA CACHE] No persistent compilation cache detected. Training will compile on-the-fly.")
    return None

XLA_CACHE_DIR = resolve_xla_cache_dir_notebook()

# Optional Overrides (None uses canonical defaults per stage/track)
DATASET_PATH = None    # Path(s) to custom verified traces JSONL, or None for canonical
OUTPUT_DIR = f"data/models/qwen3_5_9b_sft_{TRACK}_{STAGE}_{PRECISION}__{HF_RUN_TAG}"  # Isolate local state too

# ---- Every value below is passed verbatim to the training command ----
CANONICAL_RESIZE = True   # --canonical-resize: aspect-preserving FULL / CROP / COMPARE views (tools still run on the native image)
CANONICAL_LETTERBOX = IS_TPU   # --canonical-letterbox (only with CANONICAL_RESIZE): True = pad each canonical view with black bars to the fixed canvas, which the TPU's
                               # static vision slots need. False = scale to fit and ADD NOTHING: the rule on every GPU (RTX 4090, A100, any CUDA; QLoRA or LoRA).
                               # GPU: the script refuses True. For the full image at its native size instead, set CANONICAL_RESIZE = False. Keep SFT, GRPO and eval identical.
TARGET_SEQ_LEN = 32768 if HARDWARE_PROFILE == "tpu" else 16384    # --max-seq-len. The tpu profile pads every sample to this length (static graph) and its 19 vision slots add 7,488 tokens to each one, so it needs the larger value; GPU profiles keep 16384.

# ---- TPU only (not passed on GPU) ----
# AOT ladder: one static graph per length, compiled in ascending order and recorded one by one in the run report, so a single
# queue wait shows the largest length that fits one 16 GiB chip. () compiles only TARGET_SEQ_LEN.
TPU_AOT_BUCKETS = (16384, 32768) if IS_TPU else ()
if TPU_AOT_BUCKETS and (list(TPU_AOT_BUCKETS) != sorted(set(TPU_AOT_BUCKETS)) or TARGET_SEQ_LEN not in TPU_AOT_BUCKETS):
    raise ValueError("TPU_AOT_BUCKETS must be ascending, unique and contain TARGET_SEQ_LEN")
# Run report: written to TPU_REPORT_DIR as it happens and pushed to TPU_REPORT_HUB_REPO while the run is alive, so a killed or
# crashed session still leaves evidence. "" disables. See docs/TPU_DIAGNOSTICS.md.
TPU_REPORT_DIR = "/kaggle/working/tpu_report" if IS_TPU else ""
TPU_REPORT_HUB_REPO = HF_REPO if IS_TPU else ""    # an existing repo (never created); "" keeps the report local
TPU_REPORT_HEARTBEAT_SECONDS = 60
TPU_REPORT_STACK_SECONDS = 900       # dump every thread's Python stack this often (shows where a long compile is stuck)
TPU_REPORT_TPU_INFO_SECONDS = 300    # per-chip HBM from the tpu-info CLI
TPU_REPORT_STEP_SNAPSHOT_EVERY = 25  # XLA compile count / CPU fallbacks / memory every N optimizer steps (steps 1-3 always)
TPU_REPORT_UPLOAD_SECONDS = 600

def tpu_report_flags(role):
    if not TPU_REPORT_DIR:
        return []
    flags = ["--tpu-report-dir", f"{TPU_REPORT_DIR}/{role}",
             "--tpu-report-heartbeat-seconds", str(TPU_REPORT_HEARTBEAT_SECONDS),
             "--tpu-report-stack-seconds", str(TPU_REPORT_STACK_SECONDS),
             "--tpu-report-tpu-info-seconds", str(TPU_REPORT_TPU_INFO_SECONDS),
             "--tpu-report-step-snapshot-every", str(TPU_REPORT_STEP_SNAPSHOT_EVERY),
             "--tpu-report-upload-seconds", str(TPU_REPORT_UPLOAD_SECONDS)]
    if TPU_REPORT_HUB_REPO:
        flags += ["--tpu-report-hub-repo", TPU_REPORT_HUB_REPO,
                  "--tpu-report-hub-subfolder", f"tpu_reports/{TRACK}_{STAGE}/{HF_RUN_TAG}/{role}"]
    return flags

# ---- TPU only (not passed on GPU) ----
PAD_VISION_TO_SLOTS = True   # --pad-vision-to-slots: static vision slots, one XLA graph
VISION_SLOTS = (5, 10, 4)    # --vision-slots FULL CROP COMPARE. TRACK 'no_tools' needs only (1, 0, 0) = 1,152 vision tokens; (5, 10, 4) pads 6,336 more
TRIANGULAR_SHIM = True       # --triangular-shim (False = native torch.linalg.solve_triangular, to compare memory)

# Expected Checkpoint & Manifest Table
target_checkpoint = OUTPUT_DIR or f"data/models/qwen3_5_9b_sft_{TRACK}_{STAGE}_{PRECISION}"
# Identity is independent of a custom local OUTPUT_DIR.
hf_target_folder = f"sft/qwen3_5_9b_sft_{TRACK}_{STAGE}_{PRECISION}" + (f"__{HF_RUN_TAG}" if HF_RUN_TAG else "")

global_batch_size = BATCH_SIZE * GRAD_ACCUM_STEPS * NUM_CORES
approx_train_samples = 670
estimated_total_steps = int((approx_train_samples / NUM_CORES / GRAD_ACCUM_STEPS) * EPOCHS)
estimated_warmup_steps = max(int(estimated_total_steps * WARMUP_RATIO), 1)

manifest_data = [
    ["Curriculum Stage", STAGE],
    ["Training Track", TRACK],
    ["Base Model ID", MODEL_ID],
    ["Vision LoRA Adapter", "projector (vision merger)" if LORA_TARGET_VISION == "projector" else "none"],
    ["Gated-DeltaNet LoRA", "on (in_proj_qkv, in_proj_z, out_proj)" if LORA_LINEAR_ATTN else "off (attention LoRA on full-attention layers only)"],
    ["Vision Views", ("Canonical, aspect-preserving, black-bar padded (tools run on the native image)" if CANONICAL_LETTERBOX else "Canonical size, aspect-preserving, nothing added (tools run on the native image)") if CANONICAL_RESIZE else "Native resolution: the full image, only rounded to the processor's 32-pixel grid"],
    ["Static Vision-Slot Padding", f"{VISION_SLOTS} (FULL, CROP, COMPARE)" if (IS_TPU and PAD_VISION_TO_SLOTS) else "Off (dynamic padding)"],
    ["solve_triangular Shim", ("on" if TRIANGULAR_SHIM else "off (native solver)") if IS_TPU else "n/a (TPU only)"],
    ["Hardware profile", HARDWARE_PROFILE],
    ["Precision", PRECISION],
    ["QLoRA compute dtype", QLORA_COMPUTE_DTYPE if PRECISION == "qlora" else "n/a (unquantized base weights)"],
    ["Loss output-head chunk", LOSS_CHUNK_TOKENS],
    ["Static TPU loss chunk", TPU_LOSS_CHUNK_TOKENS],
    ["TPU AOT ladder", list(TPU_AOT_BUCKETS) if TPU_AOT_BUCKETS else "target length only"],
    ["TPU run report", (f"{TPU_REPORT_DIR} -> " + (TPU_REPORT_HUB_REPO or "local only")) if TPU_REPORT_DIR else "Off"],
    ["Hub base-model metadata", HF_BASE_MODEL_ID],
    ["Hub startup write check", HF_UPLOAD_CHECK],
    ["Attention backend", ATTN_IMPLEMENTATION],
    ["SDPA KV repetition", SDPA_REPEAT_KV],
    ["Loader", f"{LOADER_MODE}; CPU threads during loading={LOAD_CPU_THREADS}"],
    ["Debug optimizer windows", DEBUG_STEPS],
    ["Global Effective Batch Size", f"{global_batch_size} ({BATCH_SIZE} per core x {GRAD_ACCUM_STEPS} accum x {NUM_CORES} cores)"],
    ["Peak Learning Rate", f"{LEARNING_RATE} (Cosine with {int(WARMUP_RATIO*100)}% warmup, ~{estimated_warmup_steps} warmup steps)"],
    ["Estimated Total Steps", f"~{estimated_total_steps} optimization steps across {EPOCHS} epochs"],
    ["Gradient Clipping", f"max_norm = {MAX_GRAD_NORM}"],
    ["LoRA Config", f"r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}"],
    ["Max Sequence Length", f"{TARGET_SEQ_LEN} tokens" + (" (static, single XLA graph)" if IS_TPU else " (dynamic padding)")],
    ["XLA Pallas FlashAttention", str(XLA_PALLAS)],
    ["XLA SPMD Sharding", str(XLA_SPMD)],
    ["XLA Persistent Cache", XLA_CACHE_DIR if XLA_CACHE_DIR else "None (On-The-Fly Compilation)"],
    ["Validation Strategy", f"{EVAL_STRATEGY} (eval-every-steps={EVAL_EVERY_STEPS})"],
    ["Checkpoint Strategy", f"{SAVE_STRATEGY} (push-every-steps={PUSH_EVERY_STEPS})"],
    ["Target Local Checkpoint", target_checkpoint],
    ["HF Models Repository", HF_REPO if HF_REPO else "Disabled"],
    ["HF Checkpoint Subfolder", hf_target_folder],
    ["TPU Multi-Core Topology", f"{NUM_CORES} cores (Single-Process GSPMD)" if (IS_TPU and XLA_SPMD) else (f"{NUM_CORES} cores (xmp.spawn)" if IS_TPU else "Single device (1 core/GPU)")],
    ["FSDP Parameter Sharding", f"GSPMD Sharding (~2.3 GB BF16 weights / core, zero FP32 CPU overhead)" if (IS_TPU and XLA_SPMD) else (f"Enabled (~4.5 GB FP32 base weights / core)" if IS_TPU and USE_FSDP else "Disabled (DDP / Single)")],
    ["Resume from HF", str(RESUME)],
]

print("=" * 75)
print("VLM-DENTAL: STAGE 1 SFT EXECUTION MANIFEST")
print("=" * 75)
print(tabulate(manifest_data, headers=["Configuration Parameter", "Assigned Value"], tablefmt="fancy_grid"))


Qwen3.5-9B source: env -> /home/nadimi/VLM-DENTAL/data/models/Qwen3.5-9B
[XLA CACHE] No persistent compilation cache detected. Training will compile on-the-fly.
VLM-DENTAL: STAGE 1 SFT EXECUTION MANIFEST
╒═════════════════════════════╤══════════════════════════════════════════════════════════════════════════════════╕
│ Configuration Parameter     │ Assigned Value                                                                   │
╞═════════════════════════════╪══════════════════════════════════════════════════════════════════════════════════╡
│ Curriculum Stage            │ multicohort_all                                                                  │
├─────────────────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ Training Track              │ with_tools                                                                       │
├─────────────────────────────┼───────────────────────────────────────────────────────────────────────────

In [14]:

# ============================================================================
# HARDWARE-AWARE CUDA KERNEL DEPENDENCIES
#   Kaggle dual_t4  -> install cached wheel from rezanadimi dataset
#   Other CUDA GPUs -> REUSE installed causal-conv1d; compile only if missing
#   TPU             -> skip causal-conv1d and CUDA-only requirements
# ============================================================================
import os
import re
import sys
import json
import shutil
import subprocess
import tempfile
import importlib.metadata
from pathlib import Path

import torch
from torch.utils.cpp_extension import CUDA_HOME

from packaging.version import Version

CAUSAL_CONV1D_VERSION = "1.7.0"

# This dataset belongs to rezanadimi, NOT rezanadimikj.
CAUSAL_CONV1D_T4_DATASET_ID = "rezanadimi/causal-conv1d-t4-wheelhouse"
CAUSAL_CONV1D_T4_DATASET_SLUG = "causal-conv1d-t4-wheelhouse"


def _run_install(args, env=None):
    print("[DEPENDENCY]", " ".join(map(str, args)), flush=True)
    subprocess.check_call(args, env=env)


def _is_causal_conv_requirement(line):
    stripped = line.strip()
    if not stripped or stripped.startswith("#"):
        return False
    return re.match(
        r"^causal[-_]conv1d(?:\s|[<=>!~;\[]|$)",
        stripped,
        re.IGNORECASE,
    ) is not None


def _install_gpu_requirements_without_causal_conv():
    req_file = Path("requirements-gpu.txt")
    if not req_file.is_file():
        raise FileNotFoundError(
            "requirements-gpu.txt is missing from the VLM-DENTAL checkout."
        )

    lines = req_file.read_text(encoding="utf-8").splitlines()
    filtered = [
        line for line in lines
        if not _is_causal_conv_requirement(line)
    ]

    removed = [
        line for line in lines
        if _is_causal_conv_requirement(line)
    ]

    if removed:
        print("[DEPENDENCY] Deferring causal-conv1d:", removed)

    tmp_req = (
        Path(tempfile.gettempdir())
        / "vlm_dental_requirements_gpu_without_causal_conv1d.txt"
    )
    tmp_req.write_text("\n".join(filtered) + "\n", encoding="utf-8")

    if filtered:
        _run_install([
            sys.executable, "-m", "pip", "install",
            "-q", "-r", str(tmp_req)
        ])


def _verify_causal_conv1d():
    """Verify installed package version and real CUDA forward/backward."""
    from causal_conv1d import causal_conv1d_fn

    installed = importlib.metadata.version("causal-conv1d")

    # Accept official wheel local version suffixes such as +cu12torch...
    if Version(installed).base_version != CAUSAL_CONV1D_VERSION:
        raise RuntimeError(
            f"Expected causal-conv1d {CAUSAL_CONV1D_VERSION}; "
            f"found {installed}."
        )

    for gpu in range(torch.cuda.device_count()):
        device = f"cuda:{gpu}"

        x = torch.randn(
            2, 64, 128,
            device=device,
            dtype=torch.float16,
            requires_grad=True,
        )
        weight = torch.randn(
            64, 4,
            device=device,
            dtype=torch.float16,
            requires_grad=True,
        )

        y = causal_conv1d_fn(x, weight)

        assert y.shape == x.shape
        assert torch.isfinite(y).all()

        y.float().square().mean().backward()

        assert x.grad is not None
        assert weight.grad is not None

        torch.cuda.synchronize(gpu)

        print(
            f"[DEPENDENCY OK] GPU {gpu}: "
            "causal-conv1d forward/backward passed."
        )

    print(f"[DEPENDENCY OK] causal-conv1d={installed}")
    return installed


def _find_t4_wheelhouse():
    input_root = Path("/kaggle/input")

    candidates = [
        input_root / "datasets" / "rezanadimi" / CAUSAL_CONV1D_T4_DATASET_SLUG,
        input_root / CAUSAL_CONV1D_T4_DATASET_SLUG,
    ]

    if input_root.is_dir():
        candidates.extend(
            p for p in input_root.rglob(CAUSAL_CONV1D_T4_DATASET_SLUG)
            if p.is_dir()
        )

    seen = set()

    for root in candidates:
        key = str(root.resolve()) if root.exists() else str(root)

        if key in seen:
            continue
        seen.add(key)

        if root.is_dir():
            wheels = sorted(
                root.rglob(
                    f"causal_conv1d-{CAUSAL_CONV1D_VERSION}*.whl"
                )
            )
            if wheels:
                return root, wheels

    raise FileNotFoundError(
        "Cached T4 wheelhouse not found. Attach: "
        f"{CAUSAL_CONV1D_T4_DATASET_ID}"
    )


def _compile_causal_conv1d():
    """Source-build only when the installed extension is unavailable."""
    nvcc = (
        shutil.which("nvcc")
        or (
            str(Path(CUDA_HOME) / "bin" / "nvcc")
            if CUDA_HOME else None
        )
    )

    if not nvcc or not Path(nvcc).is_file():
        raise RuntimeError(
            "causal-conv1d is missing/unusable and nvcc is unavailable."
        )

    # Prefer a compiler that actually accepts C++20.
    compiler = None

    probe_source = """
#include <type_traits>
#include <optional>
#include <string_view>
int main() { return 0; }
"""

    for name in ("g++-12", "g++-11", "g++", "c++"):
        candidate = shutil.which(name)
        if not candidate:
            continue

        probe = subprocess.run(
            [candidate, "-std=c++20", "-x", "c++",
             "-fsyntax-only", "-"],
            input=probe_source,
            text=True,
            capture_output=True,
        )

        if probe.returncode == 0:
            compiler = candidate
            break

    if compiler is None:
        raise RuntimeError(
            "Cannot compile causal-conv1d: no installed C++ compiler "
            "supports -std=c++20. Install GCC/G++ 11 or newer. "
            "On the RTX 8000 server, an already-working installed "
            "causal-conv1d should be reused instead."
        )

    env = os.environ.copy()
    env["CC"] = compiler
    env["CXX"] = compiler
    env["CUDAHOSTCXX"] = compiler
    env["MAX_JOBS"] = "2"

    archs = sorted({
        f"{torch.cuda.get_device_capability(i)[0]}."
        f"{torch.cuda.get_device_capability(i)[1]}"
        for i in range(torch.cuda.device_count())
    })

    env["TORCH_CUDA_ARCH_LIST"] = ";".join(archs)

    print("[DEPENDENCY] C++ compiler:", compiler)
    print("[DEPENDENCY] CUDA compiler:", nvcc)
    print("[DEPENDENCY] CUDA architectures:", archs)

    _run_install([
        sys.executable, "-m", "pip", "install",
        "-q", "ninja", "packaging", "wheel",
    ])

    _run_install([
        sys.executable, "-m", "pip", "install",
        "--no-build-isolation",
        "--no-deps",
        "--no-cache-dir",
        "--no-binary=causal-conv1d",
        f"causal-conv1d=={CAUSAL_CONV1D_VERSION}",
    ], env=env)

    _verify_causal_conv1d()


# ============================================================================
# MAIN HARDWARE DISPATCH
# ============================================================================

if HARDWARE_PROFILE == "tpu":

    print(
        "[DEPENDENCY] TPU: skipping causal-conv1d "
        "and all CUDA-only GPU requirements."
    )

    if not IS_TPU:
        raise RuntimeError(
            "HARDWARE_PROFILE='tpu' but runtime is not TPU."
        )

elif torch.cuda.is_available():

    if IS_TPU:
        raise RuntimeError("Conflicting CUDA and TPU detection.")

    if HARDWARE_PROFILE == "dual_t4":

        if torch.cuda.device_count() != 2:
            raise RuntimeError("dual_t4 requires exactly two GPUs.")

        for i in range(2):
            name = torch.cuda.get_device_name(i)
            capability = torch.cuda.get_device_capability(i)

            if "T4" not in name or capability != (7, 5):
                raise RuntimeError(
                    f"Expected T4 sm_75; got {name}, {capability}."
                )

    _install_gpu_requirements_without_causal_conv()

    if HARDWARE_PROFILE == "dual_t4" and IS_KAGGLE:

        # ------------------------------------------------------------
        # Kaggle dual T4: load the prebuilt wheel, never compile.
        # ------------------------------------------------------------
        wheel_root, wheels = _find_t4_wheelhouse()

        if len(wheels) != 1:
            raise RuntimeError(
                f"Expected exactly one wheel; found {wheels}"
            )

        wheel = wheels[0]
        manifest = wheel.parent / "causal_conv1d_build_info.json"

        if not manifest.is_file():
            raise FileNotFoundError(f"Missing build manifest: {manifest}")

        built = json.loads(manifest.read_text(encoding="utf-8"))

        current = {
            "python_tag": f"cp{sys.version_info.major}{sys.version_info.minor}",
            "torch_version": torch.__version__,
            "torch_cuda": torch.version.cuda,
            "cxx11_abi": bool(torch._C._GLIBCXX_USE_CXX11_ABI),
        }

        mismatches = {
            k: (built.get(k), v)
            for k, v in current.items()
            if built.get(k) != v
        }

        if built.get("cuda_arch_list") != "7.5":
            mismatches["cuda_arch_list"] = (
                built.get("cuda_arch_list"), "7.5"
            )

        if mismatches:
            raise RuntimeError(
                f"Cached T4 wheel incompatible: {mismatches}"
            )

        print(f"[DEPENDENCY] Dataset: {CAUSAL_CONV1D_T4_DATASET_ID}")
        print(f"[DEPENDENCY] Cached wheel: {wheel.name}")

        _run_install([
            sys.executable, "-m", "pip", "install",
            "--no-deps", "--no-index", str(wheel),
        ])

        _verify_causal_conv1d()

    else:

        # ------------------------------------------------------------
        # OTHER CUDA GPUs, INCLUDING QUADRO RTX 8000:
        # First reuse the already-installed working extension.
        # Absolutely do not force-reinstall it.
        # ------------------------------------------------------------
        try:
            existing_version = importlib.metadata.version(
                "causal-conv1d"
            )

            print(
                f"[DEPENDENCY] Found existing causal-conv1d "
                f"{existing_version}; checking CUDA kernels..."
            )

            _verify_causal_conv1d()

            print(
                "[DEPENDENCY] Reusing existing installation. "
                "No compilation or reinstallation required."
            )

        except importlib.metadata.PackageNotFoundError:
            print(
                "[DEPENDENCY] causal-conv1d is not installed. "
                "Building for this GPU."
            )
            _compile_causal_conv1d()

        except Exception as exc:
            # Existing package is present but unusable.
            raise RuntimeError(
                "Existing causal-conv1d failed its compatibility or "
                "CUDA forward/backward test. No automatic force-reinstall "
                "was performed. Diagnose the existing installation first. "
                f"Original error: {exc}"
            ) from exc

else:
    raise RuntimeError(
        f"HARDWARE_PROFILE={HARDWARE_PROFILE!r}: "
        "neither CUDA nor TPU detected."
    )

print("[DEPENDENCY] Hardware-aware dependency setup complete.")


[DEPENDENCY] /home/nadimi/VLM-DENTAL/.venv/bin/python -m pip install -q -r /tmp/vlm_dental_requirements_gpu_without_causal_conv1d.txt
[DEPENDENCY] Found existing causal-conv1d 1.7.0; checking CUDA kernels...
[DEPENDENCY OK] GPU 0: causal-conv1d forward/backward passed.
[DEPENDENCY OK] causal-conv1d=1.7.0
[DEPENDENCY] Reusing existing installation. No compilation or reinstallation required.
[DEPENDENCY] Hardware-aware dependency setup complete.


## 5a. Token-Length Manifest (run once per `CANONICAL_RESIZE` setting)

The length filter in `DentalSFTDataset` needs a manifest measured with the same `CANONICAL_RESIZE` setting as training. A manifest from the other mode (or measured with the other `CANONICAL_LETTERBOX`) is ignored with a warning; after changing either, recompute it.

In [15]:
import subprocess

RECOMPUTE_TOKEN_MANIFEST = False   # False reuses an existing manifest measured with the same CANONICAL_RESIZE
MANIFEST_PATH = f"{DATA_DIR}/traces/trace_token_lengths.json"

if RECOMPUTE_TOKEN_MANIFEST:
    m_cmd = ["python", "scripts/compute_exact_trace_lengths.py", "--model-id", MODEL_ID,
             "--data-dir", DATA_DIR, "--output-file", MANIFEST_PATH, "--recompute",
             "--vision-slots", *map(str, VISION_SLOTS)]
    if CANONICAL_RESIZE:
        m_cmd.append("--canonical-resize")
        m_cmd.append("--canonical-letterbox" if CANONICAL_LETTERBOX else "--no-canonical-letterbox")
    m_cmd_str = " ".join(m_cmd)
    print(f"[MANIFEST] {m_cmd_str}")
    !{m_cmd_str}

## 5b. Ahead-Of-Time (AOT) XLA Persistent Cache Warmup (Optional)

Pre-compiles the static sequence-length bucket computation graphs sequentially across the 8 Cloud TPU v5e-8 cores, saving compiled binaries to `/kaggle/working/xla_cache`.
Flushes compiler memory between buckets to prevent host CPU RAM spikes and eliminate the 15+ minute JIT compilation freeze during training.


In [16]:
# # =========================================================================
# # AHEAD-OF-TIME (AOT) XLA COMPILATION WARMUP
# # =========================================================================
# # Set RUN_AOT_WARMUP = True to pre-compile the persistent cache on this VM.
# # TPU only. Compiles one static graph per length in TPU_AOT_BUCKETS (or only TARGET_SEQ_LEN when it is empty).
# RUN_AOT_WARMUP = IS_TPU
# AOT_CACHE_DIR = "/kaggle/working/xla_cache"

# if RUN_AOT_WARMUP:
#     import shlex
#     # Verify backend/settings before starting the expensive full-model AOT.
#     _aot_source = Path("scripts/warmup_xla_cache.py").read_text()
#     if "--tpu-loss-chunk-tokens" not in _aot_source:
#         raise RuntimeError("This notebook needs patches 0042/0043 in the checkout. Apply/push the patch before launching.")
#     if TPU_REPORT_DIR and "tpu_diagnostics" not in _aot_source:
#         raise RuntimeError("Checkout is missing the TPU run report. Apply/push patch 0059 before launching this notebook.")
#     # Clear conflicting legacy TPU multi-host cluster address variables on Kaggle/Colab
#     for _var in ["TPU_PROCESS_ADDRESSES", "TPU_PROCESS_COUNT", "CLOUD_TPU_TASK_ID", "PJRT_DEVICE", "XLA_FLAGS", "LIBTPU_INIT_ARGS"]:
#         os.environ.pop(_var, None)

#     if RUN_TPU_LOSS_PROBE:
#         import subprocess
#         subprocess.run([
#             sys.executable, "-u", "scripts/probe_tpu_chunked_loss.py",
#             "--num-cores", str(NUM_CORES), "--chunk-tokens", str(TPU_LOSS_CHUNK_TOKENS),
#         ], check=True, timeout=600)
#         TPU_LOSS_PROBE_PASSED = (NUM_CORES, TPU_LOSS_CHUNK_TOKENS)

#     warmup_cmd = [
#         "python", "-W", "ignore::UserWarning", "scripts/warmup_xla_cache.py",
#         "--model-id", MODEL_ID,
#         "--cache-dir", AOT_CACHE_DIR,
#         "--track", TRACK,
#         "--precision", PRECISION,
#         "--num-cores", str(NUM_CORES),
#         "--max-seq-len", str(TARGET_SEQ_LEN),
#         "--pad-vision-to-slots" if PAD_VISION_TO_SLOTS else "--no-pad-vision-to-slots",
#         "--vision-slots", *map(str, VISION_SLOTS),
#         "--triangular-shim" if TRIANGULAR_SHIM else "--no-triangular-shim",
#         "--spmd" if XLA_SPMD else "--no-spmd",
#         "--fsdp" if USE_FSDP else "--no-fsdp",
#         "--attn-implementation", "sdpa",
#         "--lora-target-vision", LORA_TARGET_VISION,
#         "--lora-linear-attn" if LORA_LINEAR_ATTN else "--no-lora-linear-attn",
#     ]
#     if IS_TPU:
#         warmup_cmd += ["--tpu-loss-chunk-tokens", str(TPU_LOSS_CHUNK_TOKENS)]
#     if TPU_AOT_BUCKETS:
#         warmup_cmd += ["--buckets", *map(str, TPU_AOT_BUCKETS)]
#     warmup_cmd += tpu_report_flags("aot")
#     # Match the actual LoRA rank/dropout and optimizer LR used in training.
#     warmup_cmd += ["--lora-r", str(LORA_R), "--lora-alpha", str(LORA_ALPHA),
#                    "--lora-dropout", str(LORA_DROPOUT), "--learning-rate", str(LEARNING_RATE)]
#     if XLA_PALLAS:
#         warmup_cmd.append("--xla-pallas")
#     else:
#         warmup_cmd.append("--no-xla-pallas")

#     warmup_cmd_str = shlex.join(warmup_cmd)
#     print(f"[LAUNCHING AOT WARMUP] Executing:\n{warmup_cmd_str}\n")
#     # Clean up any lingering TPU device processes before launching
#     !pkill -9 -f "multiprocessing.spawn" 2>/dev/null || true
#     !pkill -9 -f "multiprocessing.resource_tracker" 2>/dev/null || true
#     !pkill -9 -f "warmup_xla_cache.py" 2>/dev/null || true
#     !pkill -9 -f "train_sft.py" 2>/dev/null || true
#     !fuser -k -9 /dev/vfio/* 2>/dev/null || true
#     !XLA_FLAGS="" LIBTPU_INIT_ARGS="" PYTHONWARNINGS="ignore::UserWarning" {warmup_cmd_str}
# else:
#     print("[INFO] RUN_AOT_WARMUP is False. Skipping pre-compilation (training will use attached cache or compile on-the-fly).")


## 5c. Export XLA Cache to Kaggle Dataset (One-Time Export)

Exports the pre-compiled `/kaggle/working/xla_cache` folder as a private Kaggle Dataset so it can be attached to future training notebooks for instant 0-second step 0 starts.


In [17]:
# # =========================================================================
# # EXPORT XLA PERSISTENT CACHE TO KAGGLE DATASETS (VIA KAGGLE CLI)
# # =========================================================================
# # Set UPLOAD_TO_KAGGLE = True to publish the generated cache directory.
# UPLOAD_TO_KAGGLE = True
# KAGGLE_DATASET_ID = "rezanadimikj/vlm-dental-xla-cache"
# CACHE_EXPORT_DIR = Path("/kaggle/working/xla_cache")

# if UPLOAD_TO_KAGGLE and CACHE_EXPORT_DIR.is_dir():
#     import json, os, sys
#     from pathlib import Path

#     # 1. Ensure dataset metadata exists for Kaggle CLI
#     meta_file = CACHE_EXPORT_DIR / "dataset-metadata.json"
#     if not meta_file.exists():
#         meta = {
#             "title": "VLM-DENTAL SFT XLA Persistent Cache",
#             "id": KAGGLE_DATASET_ID,
#             "licenses": [{"name": "apache-2.0"}],
#             "description": "Pre-compiled XLA persistent cache for VLM-DENTAL SFT on Cloud TPU v5e-8.",
#         }
#         with open(meta_file, "w", encoding="utf-8") as f:
#             json.dump(meta, f, indent=2)

#     # 2. Upload via Kaggle CLI (dir-mode tar for rapid packaging)
#     !{sys.executable} -m pip install -q kaggle
#     print(f"[KAGGLE EXPORT] Uploading {CACHE_EXPORT_DIR} to {KAGGLE_DATASET_ID}...")
#     !{sys.executable} -m kaggle datasets version -p {CACHE_EXPORT_DIR} -m "Update SFT XLA persistent cache" --dir-mode tar || {sys.executable} -m kaggle datasets create -p {CACHE_EXPORT_DIR} --dir-mode tar
# elif not CACHE_EXPORT_DIR.is_dir():
#     print(f"[INFO] Cache directory {CACHE_EXPORT_DIR} does not exist yet. Run Section 5b first.")
# else:
#     print("[INFO] UPLOAD_TO_KAGGLE is False. Set to True to push cache to Kaggle Datasets.")


## 5d. Hardware HBM & Chip Utilization Probes (`tpu-info`)

Kaggle TPU v5e-8 provides 16 GiB physical HBM per chip across 8 cores (128 GiB total).

### Live HBM Verification Protocol:
1. `scripts/train_sft.py` and `scripts/warmup_xla_cache.py` automatically log hardware HBM consumption (`xm.get_memory_info()`) before and after the forward and backward passes of Step 0.
2. **Parallel Live Monitoring**: Run `!tpu-info` in a separate notebook cell or terminal while Step 0 is executing. It queries the physical chips directly and prints the per-chip table (Chip 0 to Chip 7) showing true per-core HBM usage.
3. **Expected HBM Range**: The vision tower's activations are bounded to ~1.35 GiB per chip under canonical FULL scans (1536×768) with 0 saved backward activations (via `torch.no_grad()`). Total per-chip HBM (including LLM backbone activations and weights) should sit comfortably below the 16 GiB physical ceiling with zero chips spiking toward it.

In [18]:
# # Run tpu-info to inspect physical per-chip memory and utilization across all 8 TPU cores
# try:
#     !tpu-info
# except Exception as e:
#     print(f"[TPU-INFO] Could not run tpu-info: {e}")


## 6. Launch SFT Training Pipeline



In [19]:
DEBUG_STEPS = 0

In [ ]:
import os
import sys
import shlex
import subprocess

# Catch a stale clone before expensive loading; no embedded runtime monkey patches.
_train_source = Path("scripts/train_sft.py").read_text()
for _flag in ("--hub-base-model-id", "--hf-run-tag", "--hf-upload-check", "--tpu-loss-chunk-tokens"):
    if _flag not in _train_source:
        raise RuntimeError(f"Checkout is missing {_flag}. Apply/push patch 0043 before launching this notebook.")
if TPU_REPORT_DIR and "tpu_diagnostics" not in _train_source:
    raise RuntimeError("Checkout is missing the TPU run report. Apply/push patch 0059 before launching this notebook.")

# Environment inherited by the probe, preflight, and training processes.
run_env = os.environ.copy()
for key in (
    "TPU_PROCESS_ADDRESSES",
    "TPU_PROCESS_COUNT",
    "CLOUD_TPU_TASK_ID",
    "PJRT_DEVICE",
):
    run_env.pop(key, None)

run_env["XLA_FLAGS"] = ""
run_env["LIBTPU_INIT_ARGS"] = ""
run_env["PYTHONWARNINGS"] = "ignore::UserWarning"

if not IS_TPU:
    run_env.setdefault(
        "PYTORCH_CUDA_ALLOC_CONF",
        "expandable_segments:True",
    )

# Attention probe: only arguments supported by probe_sft_attention.py.
# No model weights are loaded by this probe.
if RUN_ATTENTION_PROBE and not IS_TPU:
    probe_cmd = [
        sys.executable,
        "-u",
        "scripts/probe_sft_attention.py",
        "--model-id", str(MODEL_ID),
        "--precision", PRECISION,
        "--qlora-compute-dtype", QLORA_COMPUTE_DTYPE,
        "--attn-implementation", ATTN_IMPLEMENTATION,
        "--sdpa-repeat-kv", SDPA_REPEAT_KV,
    ]

    print("[ATTENTION PROBE]", shlex.join(probe_cmd), flush=True)
    subprocess.run(
        probe_cmd,
        env=run_env,
        check=True,
        timeout=180,
    )

# Actual training command: chunking and loading controls belong HERE.
cmd = [
    sys.executable,
    "-u",
    "-W", "ignore::UserWarning",
    "scripts/train_sft.py",
    "--track", TRACK,
    "--stage", STAGE,
    "--model-id", str(MODEL_ID),
    "--data-dir", str(DATA_DIR),

    "--lora-target-vision", LORA_TARGET_VISION,
    "--lora-linear-attn" if LORA_LINEAR_ATTN else "--no-lora-linear-attn",
    "--mask-malformed-turns"
    if MASK_MALFORMED_TURNS else "--no-mask-malformed-turns",
    "--dev-split", str(DEV_SPLIT),

    "--precision", PRECISION,
    "--qlora-compute-dtype", QLORA_COMPUTE_DTYPE,
    "--loss-chunk-tokens", str(LOSS_CHUNK_TOKENS),
    "--attn-implementation", ATTN_IMPLEMENTATION,
    "--sdpa-repeat-kv", SDPA_REPEAT_KV,
    "--loader-mode", LOADER_MODE,
    "--load-cpu-threads", str(LOAD_CPU_THREADS),
    "--load-traceback-seconds", str(LOAD_TRACEBACK_SECONDS),
    "--debug-steps", str(DEBUG_STEPS),

    "--batch-size", str(BATCH_SIZE),
    "--gradient-accumulation-steps",
    str(1 if DEBUG_STEPS else GRAD_ACCUM_STEPS),
    "--learning-rate", str(LEARNING_RATE),
    "--warmup-ratio", str(WARMUP_RATIO),
    "--max-grad-norm", str(MAX_GRAD_NORM),
    "--epochs", str(EPOCHS),

    "--lora-r", str(LORA_R),
    "--lora-alpha", str(LORA_ALPHA),
    "--lora-dropout", str(LORA_DROPOUT),

    "--eval-strategy", EVAL_STRATEGY,
    "--eval-every-steps", str(EVAL_EVERY_STEPS),
    "--save-strategy", SAVE_STRATEGY,
    "--push-every-steps", str(PUSH_EVERY_STEPS),
    "--max-seq-len", str(TARGET_SEQ_LEN),

    "--canonical-resize"
    if CANONICAL_RESIZE else "--no-canonical-resize",
    "--canonical-letterbox"
    if CANONICAL_LETTERBOX else "--no-canonical-letterbox",
]

cmd += ["--hub-base-model-id", HF_BASE_MODEL_ID,
        "--hf-upload-check" if HF_UPLOAD_CHECK else "--no-hf-upload-check"]
if HF_RUN_TAG:
    cmd += ["--hf-run-tag", HF_RUN_TAG]
if RUN_TPU_LOSS_PROBE and TPU_LOSS_PROBE_PASSED != (NUM_CORES, TPU_LOSS_CHUNK_TOKENS):
    subprocess.run([
        sys.executable, "-u", "scripts/probe_tpu_chunked_loss.py",
        "--num-cores", str(NUM_CORES), "--chunk-tokens", str(TPU_LOSS_CHUNK_TOKENS),
    ], env=run_env, check=True, timeout=600)
    TPU_LOSS_PROBE_PASSED = (NUM_CORES, TPU_LOSS_CHUNK_TOKENS)

if REQUIRE_EFFICIENT_SDPA:
    cmd.append("--require-efficient-sdpa")

if IS_TPU:
    cmd += [
        "--tpu-loss-chunk-tokens", str(TPU_LOSS_CHUNK_TOKENS),
        "--num-cores", str(NUM_CORES),
        "--spmd" if XLA_SPMD else "--no-spmd",
        "--fsdp" if USE_FSDP else "--no-fsdp",
        "--pad-vision-to-slots"
        if PAD_VISION_TO_SLOTS else "--no-pad-vision-to-slots",
        "--vision-slots", *map(str, VISION_SLOTS),
        "--triangular-shim"
        if TRIANGULAR_SHIM else "--no-triangular-shim",
        "--xla-pallas" if XLA_PALLAS else "--no-xla-pallas",
    ]
    if XLA_CACHE_DIR:
        cmd += ["--xla-cache-dir", str(XLA_CACHE_DIR)]
    cmd += tpu_report_flags("sft")

if DATASET_PATH:
    cmd += ["--dataset-path", str(DATASET_PATH)]

if DEBUG_STEPS:
    cmd += ["--output-dir", f"data/models/sft_debug_{HARDWARE_PROFILE}"]
else:
    if OUTPUT_DIR:
        cmd += ["--output-dir", str(OUTPUT_DIR)]
    if HF_REPO:
        cmd += ["--hf-repo", HF_REPO]
    if RESUME and HF_REPO:
        cmd += ["--resume-hf", HF_REPO]

print(
    f"[SFT] precision={PRECISION}, "
    f"compute={QLORA_COMPUTE_DTYPE}, "
    f"loss_chunk_tokens={LOSS_CHUNK_TOKENS}, "
    f"debug_steps={DEBUG_STEPS}",
    flush=True,
)
print(
    "[LAUNCHING PIPELINE] Executing:\n"
    + shlex.join(cmd)
    + "\n",
    flush=True,
)

if PREFLIGHT_FIRST:
    preflight_cmd = cmd + ["--preflight"]

    print("[PREFLIGHT] Starting read-only checks...", flush=True)

    preflight = subprocess.Popen(
        preflight_cmd,
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in preflight.stdout:
        print(line, end="", flush=True)

    return_code = preflight.wait()

    if return_code != 0:
        raise RuntimeError(
            f"[PREFLIGHT FAILED] Exit code: {return_code}. "
            "Training was not started."
        )

    print("[PREFLIGHT OK] Launching training.\n", flush=True)

# Stream training output into the notebook.
with subprocess.Popen(
    cmd,
    env=run_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    try:
        for line in process.stdout:
            print(line, end="", flush=True)

        return_code = process.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, cmd)

    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise

[ATTENTION PROBE] /home/nadimi/VLM-DENTAL/.venv/bin/python -u scripts/probe_sft_attention.py --model-id /home/nadimi/VLM-DENTAL/data/models/Qwen3.5-9B --precision fp16 --qlora-compute-dtype auto --attn-implementation sdpa --sdpa-repeat-kv auto
{
  "versions": {
    "torch": "2.13.0+cu126",
    "transformers": "5.19.0",
    "peft": "0.21.2",
    "accelerate": "1.15.0",
    "bitsandbytes": "0.50.0",
    "flash-attn": null,
    "flash-linear-attention": "0.5.2",
    "causal-conv1d": "1.7.0"
  },
  "torch_cuda": "12.6",
  "gpus": [
    {
      "index": 0,
      "name": "Quadro RTX 8000",
      "compute_capability": [
        7,
        5
      ],
      "total_gib": 47.454
    }
  ]
}
[PROBE] backend=sdpa dtype=torch.float16 Q/KV heads=16/4 head_dim=256 dropout=0.0 tokens=128
[SDPA] KV repetition policy=auto; built-in repeat_kv branch; math fallback policy unchanged
[PASS GPU 0] Quadro RTX 8000 sdpa forward/backward; peak allocated=0.014 GiB
[PROBE COMPLETE] Small kernel probe passed. Full 

## 7. Training Loss & Convergence Visualizer

Plots conversational assistant training loss and held-out validation loss curves, reporting initial, best validation, and final loss metrics.

In [ ]:
import json
import matplotlib.pyplot as plt

output_dir = (OUTPUT_DIR or f"data/models/qwen3_5_9b_sft_{TRACK}_{STAGE}_{PRECISION}")
log_file = f"{output_dir}/training_loss.jsonl"

if os.path.exists(log_file):
    steps, train_losses, val_steps, val_losses = [], [], [], []
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                step = rec.get("step", len(steps))
                steps.append(step)
                train_losses.append(rec.get("loss", 0.0))
                if "val_loss" in rec:
                    val_steps.append(step)
                    val_losses.append(rec["val_loss"])

    plt.figure(figsize=(11, 5))
    plt.plot(steps, train_losses, label="Assistant Training Loss", color="#1f77b4", lw=1.8, alpha=0.85)
    if val_losses:
        plt.plot(val_steps, val_losses, label="Validation Loss (5% held-out)", color="#d62728", marker="o", lw=2)

    plt.title(f"VLM-DENTAL Stage 1 SFT Convergence — Stage: {STAGE.upper()} | Track: {TRACK.upper()}")
    plt.xlabel("Optimization Steps")
    plt.ylabel("Assistant Conversational Cross-Entropy Loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    best_val_str = f" | Best Val Loss: {min(val_losses):.4f}" if val_losses else ""
    print(f"[METRICS] Initial Loss: {train_losses[0]:.4f} -> Final Loss: {train_losses[-1]:.4f}{best_val_str}")
    print(f"[CHECKPOINTS] Checkpoints saved to {output_dir}")
else:
    print(f"[INFO] Log file {log_file} not found yet. Execute Cell 6 to run training.")

## 8. Multi-Condition Evaluation on DENTEX Test Split (§6, §20, §26)

Unified evaluation across baseline and fine-tuned models on the official DENTEX test split using `scripts/evaluate_models.py`.
Supports all conditions (`base_no_tools`, `base_with_tools`, `sft_no_tools`, `sft_with_tools`, or `all`), set-level multi-finding matching (Rule 13), and cached zero-shot baseline re-use.


In [ ]:

from dental_agent.harness.notebook_eval import NotebookEvaluations
import json

# Hardware evaluation profiles.
# Aliases: rtx4090 -> rtx4090_qlora; rtxpro6000_48gb -> workstation48_lora.
EVAL_PROFILES = {
    "dual_t4": {
        "tag": "dual_t4_fp16",
        "precision": "qlora",
        "compute": "fp16",
        "attention": "sdpa",
        "kv": "on",
        "loader": "staged",
        "threads": 1,
        "letterbox": False,
    },
    "rtx4090_qlora": {
        "tag": "rtx4090_bf16",
        "precision": "qlora",
        "compute": "bf16",
        "attention": "auto",
        "kv": "auto",
        "loader": "standard",
        "threads": 0,
        "letterbox": False,
    },
    "workstation48_lora": {
        "tag": "workstation48_lora_bf16",
        "precision": "bf16",
        "compute": "auto",
        "attention": "auto",
        "kv": "auto",
        "loader": "standard",
        "threads": 0,
        "letterbox": False,
    },
    "quadro_rtx8000": {
        "tag": "quadro_rtx8000_fp16",
        "precision": "fp16",
        "compute": "auto",
        "attention": "sdpa",
        "kv": "auto",
        "loader": "standard",
        "threads": 0,
        "letterbox": False,
        "serving_dtype": "float16",
    },
    "tpu": {
        "tag": "tpu_bf16",
        "precision": "bf16",
        "compute": "auto",
        "attention": "sdpa",
        "kv": "off",
        "loader": "standard",
        "threads": 0,
        "letterbox": True,
    },
    "dual_t4_fp16": {
        "tag": "dual_t4_unquantized_fp16",
        "precision": "fp16",
        "compute": "auto",
        "attention": "sdpa",
        "kv": "on",
        "loader": "staged",
        "threads": 1,
        "letterbox": False,
    },
}

# Existing evaluation-results repository.
# Checkpoints are resolved from HF_ARTIFACT_REPO.
EVAL_RESULTS_REPO = "Reza-Nadimi/vlm-dental-evaluations"
EVAL_RESULTS_REPO_TYPE = None

# Match the training track.
EVAL_TRACK = TRACK  # with_tools or no_tools

# Only currently available checkpoints.
EVAL_KINDS = ["baseline", "sft"]

# Use the active training hardware profile.
EVAL_TARGET_PROFILES = globals().get(
    "HARDWARE_PROFILE",
    "quadro_rtx8000",
)

EVAL_STAGES = STAGE
EVAL_SPLIT = "test"
EVAL_SIZES = ["canonical"]

RUN_EVALUATIONS = True

print("Evaluation configuration:")
print(json.dumps({
    "target_profiles": EVAL_TARGET_PROFILES,
    "kinds": EVAL_KINDS,
    "track": EVAL_TRACK,
    "stages": EVAL_STAGES,
    "split": EVAL_SPLIT,
    "sizes": EVAL_SIZES,
}, indent=2))

sft_evaluations = NotebookEvaluations(
    track=EVAL_TRACK,
    profiles=EVAL_PROFILES,
    results_repo=EVAL_RESULTS_REPO,
    results_repo_type=EVAL_RESULTS_REPO_TYPE,
    target_profiles=EVAL_TARGET_PROFILES,
    kinds=EVAL_KINDS,
    stages=EVAL_STAGES,
    model_id=MODEL_ID,
    data_dir=DATA_DIR,
    dev_split=DEV_SPLIT,
    split=EVAL_SPLIT,
    sizes=EVAL_SIZES,
)

# Inspect the baseline and SFT evaluation jobs.
sft_evaluations.list()

# Execute evaluations.
SFT_EVAL_STATUS = sft_evaluations.run(
    execute=RUN_EVALUATIONS
)

print(json.dumps(SFT_EVAL_STATUS, indent=2))
